# Stg Layer — restaurant_place_details (Google Places Place Details, refreshed monthly)

For every restaurant that `stg_restaurant_geocode` matched to a Google place, and (since 2026-11-01) for the **Google-only** places that `auto_search` found with Text Search **Pro** (`silver.stg_places`, no rating / price / hours / phone / website), read the **Place Details (New)** fields that the dashboard needs and keep the latest answer in `${silver}.stg_restaurant_place_details` (one row per `place_id`).

**Fields:** `rating`, `userRatingCount` (Google stars and number of reviews), `priceRange` (e.g. 200,000 → 800,000 ₫ per person), `priceLevel`, `primaryType` / `primaryTypeDisplayName` (e.g. `chinese_restaurant` / "Nhà hàng Quảng Đông", used for the cuisine), `types`, `googleMapsUri`, `websiteUri`, `nationalPhoneNumber`, `regularOpeningHours` (Google's weekday lines, added 2026-11-01: Enterprise fields, so the SKU and the price stay the same), the first photo's resource name (`photos[0].name`, for the dashboard later), and **`reviews`** (up to 5 Google reviews per place, added 2026-10-31) kept in `${silver}.stg_restaurant_place_reviews` (one row per review, accumulated over the monthly re-reads).

**Flow:**
1. Matched `place_id`s from the geocode cache (restaurants with a YouTube review) that were **never fetched** or were fetched more than `REFRESH_DAYS` ago (ratings change, so each place is re-read about once a month); never-fetched first
   Then the **top `WEEKLY_TOP_N` (20) restaurants by `legit_score`** (`gold.agg_restaurant_ranking` of the last run) read more than `WEEKLY_REFRESH_DAYS` (7) ago, so the most-watched places get a weekly rating point in `stg_place_rating_reads` (added 2026-11-01, ~65 extra calls per month, taken from the Google-only share)
   Then **Google-only** places of `stg_places` without a rating (found by Text Search Pro), never fetched or older than `GOOGLE_ONLY_REFRESH_DAYS` (90), the places found by the most keywords (`search_hits`) first, at most `GOOGLE_ONLY_DAILY_LIMIT` per day so the YouTube restaurants always get their monthly re-read
2. Call `GET places/{place_id}` within the **daily** and **monthly** caps (counted over every run in `stg_restaurant_place_details_calls`); the rest wait for the next run
   Places read before reviews were added (`reviews_read` not true) come next, so each gets its reviews once.
3. MERGE into the table (update on `place_id`); MERGE the reviews on `review_id` (new reviews inserted, seen ones refreshed, `last_seen_at` = latest read that returned them). HTTP errors are **not** stored, so they are retried next run; 404 is stored as `fetch_status = not_found`.

**Quota:** on HTTP 429 the run stops calling the API but does not fail. A rejected key (401 / 403) fails the task.

**Cost:** rating / userRatingCount / priceRange / priceLevel are **Enterprise** fields and `reviews` is an **Atmosphere** field. Google bills a call at the highest SKU in the field mask, so since 2026-10-31 each call is **Place Details Enterprise + Atmosphere** (its own 1,000 free calls per month at the time of writing; check the Google Maps pricing page before raising the cap). `MONTHLY_CALL_LIMIT = 900` keeps the bill at $0; ~140 YouTube places × 1 refresh per month use ~15% of it, Google-only places get at most 24 per day (~750 per month). Text Search in `stg_restaurant_geocode` stays Pro.

**Secret:** `maps/api_key` (Places API (New)).

In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

GEOCODE_TABLE = f"{SILVER}.stg_restaurant_geocode"
DETAILS_TABLE = f"{SILVER}.stg_restaurant_place_details"
CALLS_TABLE = f"{SILVER}.stg_restaurant_place_details_calls"  # API calls per run, for the caps
REVIEWS_TABLE = f"{SILVER}.stg_restaurant_place_reviews"     # Google reviews (<= 5 per read), one row per review
DAILY_CALL_LIMIT = 300      # Place Details calls per Vietnam day, over all runs (first backfill; the monthly cap is the real guard)
MONTHLY_CALL_LIMIT = 900    # per Vietnam calendar month: stays under the 1,000 free Enterprise + Atmosphere calls
REFRESH_DAYS = 30           # re-read a place after this many days
PLACES_TABLE = f"{SILVER}.stg_places"  # Google Maps Text Search results (Google-only places come from here)
GOOGLE_ONLY_REFRESH_DAYS = 90  # Google-only places (no YouTube review) are re-read less often
GOOGLE_ONLY_DAILY_LIMIT = 24   # Google-only reads per Vietnam day (~750 / month), after the YouTube restaurants
RANKING_TABLE = f"{GOLD}.agg_restaurant_ranking"  # legit_score of the last run (top places re-read weekly)
WEEKLY_TOP_N = 20              # top restaurants by legit_score re-read every WEEKLY_REFRESH_DAYS (rating history)
WEEKLY_REFRESH_DAYS = 7
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
# Enterprise fields (rating, userRatingCount, priceLevel, priceRange) + the Atmosphere field reviews:
# every call is billed as Place Details Enterprise + Atmosphere
FIELD_MASK = ("id,displayName,primaryType,primaryTypeDisplayName,types,rating,userRatingCount,"
              "priceLevel,priceRange,googleMapsUri,websiteUri,nationalPhoneNumber,regularOpeningHours,photos,reviews")

print(f"Geocode source: {GEOCODE_TABLE}")
print(f"Delta target: {DETAILS_TABLE} · max {DAILY_CALL_LIMIT} calls per day, {MONTHLY_CALL_LIMIT} per month")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DETAILS_TABLE} (
        place_id                   STRING,
        fetch_status               STRING,
        place_name                 STRING,
        primary_type               STRING,
        primary_type_display_name  STRING,
        place_types                ARRAY<STRING>,
        rating                     DOUBLE,
        rating_count               INT,
        price_level                STRING,
        price_start_vnd            BIGINT,
        price_end_vnd              BIGINT,
        google_maps_uri            STRING,
        website_uri                STRING,
        phone_number               STRING,
        google_opening_hours       STRING,
        photo_name                 STRING,
        review_count_read          INT,
        reviews_read               BOOLEAN,
        fetched_at                 TIMESTAMP
    )
    USING DELTA
    COMMENT 'Google Place Details per place_id (latest read, refreshed every REFRESH_DAYS). fetch_status = ok / not_found.'
""")
# Tables created before reviews (2026-10-31) or website / phone / opening hours (2026-11-01) were added
ADDED_COLS = {"review_count_read": "INT", "reviews_read": "BOOLEAN", "website_uri": "STRING",
              "phone_number": "STRING", "google_opening_hours": "STRING"}
missing = [c for c in ADDED_COLS if c not in spark.table(DETAILS_TABLE).columns]
if missing:
    spark.sql(f"ALTER TABLE {DETAILS_TABLE} ADD COLUMNS ({', '.join(f'{c} {ADDED_COLS[c]}' for c in missing)})")
    print(f"➕ {DETAILS_TABLE}: added columns {missing}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {REVIEWS_TABLE} (
        review_id          STRING,
        place_id           STRING,
        review_rank        INT,
        rating             INT,
        review_text        STRING,
        review_language    STRING,
        original_text      STRING,
        original_language  STRING,
        author_name        STRING,
        author_uri         STRING,
        publish_time       TIMESTAMP,
        relative_time      STRING,
        google_maps_uri    STRING,
        first_seen_at      TIMESTAMP,
        last_seen_at       TIMESTAMP
    )
    USING DELTA
    COMMENT 'Google reviews from Place Details (at most 5 per read, the ones Google ranks most relevant). One row per review_id (places/<id>/reviews/<id>); review_rank = position in the latest read that returned it.'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CALLS_TABLE} (
        run_ts    TIMESTAMP,
        call_date DATE,
        api_calls INT,
        stored    INT,
        errors    INT
    )
    USING DELTA
    COMMENT 'Place Details calls per run; sums per day / month enforce DAILY_CALL_LIMIT / MONTHLY_CALL_LIMIT.'
""")

# --- Places due: never fetched first, then reads without reviews (before 2026-10-31), then the oldest reads ---
todo = spark.sql(f"""
    SELECT g.place_id, d.fetched_at
    FROM (SELECT DISTINCT place_id FROM {GEOCODE_TABLE} WHERE match_status = 'matched' AND place_id IS NOT NULL) g
    LEFT JOIN {DETAILS_TABLE} d ON d.place_id = g.place_id
    WHERE d.fetched_at IS NULL OR d.fetched_at < current_timestamp() - INTERVAL {REFRESH_DAYS} DAYS
       OR (d.fetch_status = 'ok' AND NOT COALESCE(d.reviews_read, false))
    ORDER BY d.fetched_at IS NOT NULL, COALESCE(d.reviews_read, false), d.fetched_at, g.place_id
""").collect()

# --- Top WEEKLY_TOP_N by legit_score (last run's ranking): re-read weekly for the rating history ---
weekly = []
if spark.catalog.tableExists(RANKING_TABLE):
    due_ids = {r["place_id"] for r in todo}
    weekly = [r for r in spark.sql(f"""
        SELECT t.place_id, d.fetched_at
        FROM (SELECT substr(restaurant_id, 7) AS place_id, legit_score
              FROM {RANKING_TABLE}
              WHERE restaurant_id LIKE 'place:%' AND legit_score IS NOT NULL
              ORDER BY legit_score DESC, restaurant_id LIMIT {WEEKLY_TOP_N}) t
        JOIN {DETAILS_TABLE} d ON d.place_id = t.place_id AND d.fetch_status = 'ok'
        WHERE d.fetched_at < current_timestamp() - INTERVAL {WEEKLY_REFRESH_DAYS} DAYS
        ORDER BY d.fetched_at, t.legit_score DESC
    """).collect() if r["place_id"] not in due_ids]

# --- Google-only places (auto_search Text Search Pro: no rating yet): most-found first, re-read every 90 days ---
google_only = []
if spark.catalog.tableExists(PLACES_TABLE) and "search_hits" in spark.table(PLACES_TABLE).columns:
    google_only = spark.sql(f"""
        SELECT p.place_id, d.fetched_at
        FROM {PLACES_TABLE} p
        LEFT JOIN {DETAILS_TABLE} d ON d.place_id = p.place_id
        WHERE p.rating IS NULL
          AND NOT EXISTS (SELECT 1 FROM {GEOCODE_TABLE} g WHERE g.place_id = p.place_id AND g.match_status = 'matched')
          AND (d.fetched_at IS NULL OR d.fetched_at < current_timestamp() - INTERVAL {GOOGLE_ONLY_REFRESH_DAYS} DAYS)
        ORDER BY d.fetched_at IS NOT NULL, p.search_hits DESC, d.fetched_at, p.place_id
    """).collect()

# --- Caps: calls already made today / this month (Vietnam time) by earlier runs ---
today = spark.sql(f"SELECT to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}')) AS d").collect()[0]["d"]
used = spark.sql(f"""
    SELECT COALESCE(SUM(CASE WHEN call_date = DATE'{today}' THEN api_calls END), 0) AS day,
           COALESCE(SUM(api_calls), 0) AS month
    FROM {CALLS_TABLE}
    WHERE call_date >= trunc(DATE'{today}', 'MM')
""").collect()[0]
budget = max(0, min(DAILY_CALL_LIMIT - used["day"], MONTHLY_CALL_LIMIT - used["month"]))
# YouTube restaurants first; Google-only places fill what is left, at most GOOGLE_ONLY_DAILY_LIMIT per run
# (the job runs once a day; the daily and monthly caps above still bound any extra run)
todo = todo + weekly  # monthly re-reads first, then the weekly top places
google_only_budget = max(0, min(GOOGLE_ONLY_DAILY_LIMIT, budget - len(todo)))
waiting = max(0, len(todo) - budget) + max(0, len(google_only) - google_only_budget)
print(f"Places due: {len(todo) - len(weekly)} YouTube + {len(weekly)} weekly top + {len(google_only)} Google-only · calls today: {used['day']}/{DAILY_CALL_LIMIT} · "
      f"this month: {used['month']}/{MONTHLY_CALL_LIMIT} · this run may call: {budget} (Google-only: {google_only_budget})")
todo = todo[:budget] + google_only[:google_only_budget]

In [ ]:
# --- Call Place Details (New) ---
import requests

API_KEY = dbutils.secrets.get("maps", "api_key")


def money_vnd(m):
    """Money {currencyCode, units, nanos} -> whole VND (None if missing or another currency)."""
    if not m or m.get("currencyCode", "VND") != "VND":
        return None
    return int(m.get("units", 0))


def parse_time(s):
    """RFC 3339 publishTime ('2024-05-01T10:20:30.123456Z') -> naive UTC datetime, or None."""
    from datetime import datetime
    if not s:
        return None
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00")).replace(tzinfo=None)
    except ValueError:
        return None


def parse_reviews(place_id, reviews):
    """Place Details reviews[] -> rows of stg_restaurant_place_reviews (without first/last_seen_at)."""
    rows = []
    for rank, r in enumerate(reviews or [], start=1):
        if not r.get("name"):
            continue
        text, original = r.get("text") or {}, r.get("originalText") or {}
        author = r.get("authorAttribution") or {}
        rows.append({
            "review_id": r["name"], "place_id": place_id, "review_rank": rank,
            "rating": int(r["rating"]) if r.get("rating") is not None else None,
            "review_text": text.get("text"), "review_language": text.get("languageCode"),
            "original_text": original.get("text"), "original_language": original.get("languageCode"),
            "author_name": author.get("displayName"), "author_uri": author.get("uri"),
            "publish_time": parse_time(r.get("publishTime")),
            "relative_time": r.get("relativePublishTimeDescription"),
            "google_maps_uri": r.get("googleMapsUri"),
        })
    return rows


def place_details(place_id):
    resp = requests.get(
        f"https://places.googleapis.com/v1/places/{place_id}",
        params={"languageCode": "vi", "regionCode": "VN"},
        headers={"X-Goog-Api-Key": API_KEY, "X-Goog-FieldMask": FIELD_MASK},
        timeout=30,
    )
    if resp.status_code == 404:
        return {"place_id": place_id, "fetch_status": "not_found"}, None
    if resp.status_code != 200:
        return None, f"HTTP {resp.status_code}: {resp.text[:200]}"
    p = resp.json()
    price = p.get("priceRange") or {}
    photos = p.get("photos") or []
    reviews = parse_reviews(place_id, p.get("reviews"))
    return {
        "place_id": place_id, "fetch_status": "ok",
        "place_name": (p.get("displayName") or {}).get("text"),
        "primary_type": p.get("primaryType"),
        "primary_type_display_name": (p.get("primaryTypeDisplayName") or {}).get("text"),
        "place_types": sorted(p.get("types", [])),
        "rating": float(p["rating"]) if p.get("rating") is not None else None,  # 4 comes back as an int
        "rating_count": p.get("userRatingCount"),
        "price_level": p.get("priceLevel"),
        "price_start_vnd": money_vnd(price.get("startPrice")),
        "price_end_vnd": money_vnd(price.get("endPrice")),
        "google_maps_uri": p.get("googleMapsUri"),
        "website_uri": p.get("websiteUri"),
        "phone_number": p.get("nationalPhoneNumber"),
        "google_opening_hours": "; ".join((p.get("regularOpeningHours") or {}).get("weekdayDescriptions") or []) or None,
        "photo_name": photos[0].get("name") if photos else None,
        "review_count_read": len(reviews), "reviews_read": True,
        "reviews": reviews,
    }, None


results, errors, api_calls = [], [], 0
for row in todo[:budget]:
    result, error = place_details(row["place_id"])
    api_calls += 1
    if error and error.startswith(("HTTP 401", "HTTP 403")):
        raise RuntimeError(f"Places API rejected the key (check secret maps/api_key and the API restrictions): {error}")
    if error and error.startswith("HTTP 429"):
        errors.append(f"{row['place_id']}: {error}")
        print(f"⏸️ Places API quota reached after {api_calls} call(s); the rest waits for the next run")
        break
    if error:
        errors.append(f"{row['place_id']}: {error}")
    else:
        results.append(result)
print(f"API calls: {api_calls} · stored: {len(results)} · errors (retried next run): {len(errors)}")
for e in errors[:5]:
    print("  ⚠️", e[:300])
if waiting:
    print(f"⏸️ Cap: {waiting} place(s) wait for a later run")

# Record the calls first, so the caps hold even if a later step fails
spark.sql(f"""
    INSERT INTO {CALLS_TABLE}
    VALUES (current_timestamp(), DATE'{today}', {api_calls}, {len(results)}, {len(errors)})
""")

In [ ]:
# --- MERGE into the table (latest read wins) ---
from pyspark.sql.functions import current_timestamp

schema = spark.table(DETAILS_TABLE).schema
if results:
    cols = [f for f in schema if f.name != "fetched_at"]
    rows = [{f.name: r.get(f.name) for f in cols} for r in results]
    new_df = spark.createDataFrame(rows, schema=", ".join(f"{f.name} {f.dataType.simpleString()}" for f in cols))
    new_df.withColumn("fetched_at", current_timestamp()).createOrReplaceTempView("details_src")
    spark.sql(f"""
        MERGE INTO {DETAILS_TABLE} AS t
        USING details_src AS s ON t.place_id = s.place_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
print(f"✅ Merged {len(results)} reads into {DETAILS_TABLE}")

# --- Reviews: MERGE on review_id (accumulates over the monthly re-reads) ---
review_rows = [rv for r in results for rv in r.get("reviews", [])]
if review_rows:
    rcols = [f for f in spark.table(REVIEWS_TABLE).schema if f.name not in ("first_seen_at", "last_seen_at")]
    rv_df = spark.createDataFrame([{f.name: r.get(f.name) for f in rcols} for r in review_rows],
                                  schema=", ".join(f"{f.name} {f.dataType.simpleString()}" for f in rcols))
    # Google returns one review once per read; keep one row per review_id in case of a repeat
    rv_df.dropDuplicates(["review_id"]).createOrReplaceTempView("reviews_src")
    spark.sql(f"""
        MERGE INTO {REVIEWS_TABLE} AS t
        USING reviews_src AS s ON t.review_id = s.review_id
        WHEN MATCHED THEN UPDATE SET
            t.review_rank = s.review_rank, t.rating = s.rating, t.review_text = s.review_text,
            t.review_language = s.review_language, t.original_text = s.original_text,
            t.original_language = s.original_language, t.author_name = s.author_name, t.author_uri = s.author_uri,
            t.publish_time = s.publish_time, t.relative_time = s.relative_time, t.google_maps_uri = s.google_maps_uri,
            t.last_seen_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (review_id, place_id, review_rank, rating, review_text, review_language,
            original_text, original_language, author_name, author_uri, publish_time, relative_time, google_maps_uri,
            first_seen_at, last_seen_at)
        VALUES (s.review_id, s.place_id, s.review_rank, s.rating, s.review_text, s.review_language, s.original_text,
            s.original_language, s.author_name, s.author_uri, s.publish_time, s.relative_time, s.google_maps_uri,
            current_timestamp(), current_timestamp())
    """)
print(f"✅ Merged {len(review_rows)} reviews into {REVIEWS_TABLE}")

In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT place_id) AS ids,
           COUNT_IF(rating IS NOT NULL) AS with_rating,
           COUNT_IF(price_start_vnd IS NOT NULL) AS with_price_range,
           COUNT_IF(price_level IS NOT NULL) AS with_price_level,
           COUNT_IF(photo_name IS NOT NULL) AS with_photo,
           COUNT_IF(reviews_read) AS with_reviews_read
    FROM {DETAILS_TABLE}
""").collect()[0]
rv = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT review_id) AS ids, COUNT(DISTINCT place_id) AS places FROM {REVIEWS_TABLE}").collect()[0]
print(f"📊 {DETAILS_TABLE}: {stats['n']} places · rating {stats['with_rating']} · price range {stats['with_price_range']} "
      f"· price level {stats['with_price_level']} · photo {stats['with_photo']} · reviews read {stats['with_reviews_read']}")
print("📊 Grain check (place_id): " + ("✅" if stats["n"] == stats["ids"] else "❌ duplicates"))
print(f"📊 {REVIEWS_TABLE}: {rv['n']} reviews of {rv['places']} places · grain check (review_id): " + ("✅" if rv["n"] == rv["ids"] else "❌ duplicates"))
display(spark.sql(f"""
    SELECT primary_type, primary_type_display_name, COUNT(*) AS n
    FROM {DETAILS_TABLE} GROUP BY 1, 2 ORDER BY n DESC
"""))
dbutils.jobs.taskValues.set("api_calls", api_calls)
print("\n--- stg_restaurant_place_details complete ---")